# 7. ETL - Punjenje dimenzijskog modela

Ovaj notebook izvršava ETL (Extract, Transform, Load) proces:
1. Čisti postojeće podatke u dimenzijskim tablicama
2. Puni dimenzijske tablice (`dim_projekt`, `dim_tehnicar`, `dim_prioritet_status`, `dim_vrijeme`)
3. Transformira i puni tablicu činjenica (`fact_support_tickets`)

**Preduvjeti:** Pokrenuti notebook `6_dimensional_ddl.ipynb` za kreiranje tablica.

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

In [ ]:
# 1. Postavke povezivanja
USER = 'root'
PASSWORD = '3k0p13!4'
HOST = 'localhost'
DB_NAME = 'fipu_srp_projekt'

engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DB_NAME}")
print(f"Spojeno na bazu: {DB_NAME}")

In [ ]:
# 2. Pražnjenje tablica (redoslijed: fact -> dimenzije)
with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.execute(text("TRUNCATE TABLE fact_support_tickets;"))
    conn.execute(text("TRUNCATE TABLE dim_vrijeme;"))
    conn.execute(text("TRUNCATE TABLE dim_projekt;"))
    conn.execute(text("TRUNCATE TABLE dim_tehnicar;"))
    conn.execute(text("TRUNCATE TABLE dim_prioritet_status;"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    conn.commit()
    print("Tablice su ispražnjene i spremne za novi unos.")

In [ ]:
# 3. EXTRACT - Učitavanje izvornih podataka
df = pd.read_csv('Support_tickets_PROCESSED.csv')
print(f"Učitano {len(df)} redaka iz CSV datoteke.")

# Pretvorba datumskih stupaca
df['issue_created'] = pd.to_datetime(df['issue_created'], format='ISO8601')
df['issue_resolution_date'] = pd.to_datetime(df['issue_resolution_date'], format='ISO8601')
print(f"Raspon datuma: {df['issue_created'].min()} - {df['issue_created'].max()}")

## 4. TRANSFORM & LOAD - Punjenje dimenzija

In [ ]:
# 4a. DIM_PROJEKT
dim_projekt = df[['issue_proj']].drop_duplicates().reset_index(drop=True)
dim_projekt.columns = ['naziv_projekta']
dim_projekt.to_sql('dim_projekt', engine, if_exists='append', index=False)
print(f"dim_projekt: {len(dim_projekt)} redaka uneseno.")

# Dohvati generirane ključeve
db_projekti = pd.read_sql("SELECT projekt_key, naziv_projekta FROM dim_projekt", engine)
print(db_projekti)

In [ ]:
# 4b. DIM_TEHNICAR
# Kombiniraj sve unikatne osobe (reporteri + assigneei)
reporters = df[['issue_reporter']].rename(columns={'issue_reporter': 'ime_prezime'})
assignees = df[['issue_assignee']].rename(columns={'issue_assignee': 'ime_prezime'})
dim_tehnicar = pd.concat([reporters, assignees]).drop_duplicates().dropna().reset_index(drop=True)

dim_tehnicar.to_sql('dim_tehnicar', engine, if_exists='append', index=False)
print(f"dim_tehnicar: {len(dim_tehnicar)} redaka uneseno.")

# Dohvati generirane ključeve
db_tehnicari = pd.read_sql("SELECT tehnicar_key, ime_prezime FROM dim_tehnicar", engine)
print(f"Primjer: {db_tehnicari.head(3).to_dict('records')}")

In [ ]:
# 4c. DIM_PRIORITET_STATUS
dim_ps = df[['issue_priority', 'issue_status']].drop_duplicates().reset_index(drop=True)
dim_ps.columns = ['razina_prioriteta', 'naziv_statusa']
dim_ps.to_sql('dim_prioritet_status', engine, if_exists='append', index=False)
print(f"dim_prioritet_status: {len(dim_ps)} redaka uneseno.")

# Dohvati generirane ključeve
db_ps = pd.read_sql(
    "SELECT prioritet_status_key, razina_prioriteta, naziv_statusa FROM dim_prioritet_status",
    engine
)
print(f"Primjer: {db_ps.head(3).to_dict('records')}")

In [ ]:
# 4d. DIM_VRIJEME
dates = pd.to_datetime(df['issue_created']).dt.date.unique()
dim_vrijeme = pd.DataFrame({'vrijeme_key': dates})
dim_vrijeme['vrijeme_key'] = pd.to_datetime(dim_vrijeme['vrijeme_key'])
dim_vrijeme['dan'] = dim_vrijeme['vrijeme_key'].dt.day
dim_vrijeme['mjesec'] = dim_vrijeme['vrijeme_key'].dt.month
dim_vrijeme['godina'] = dim_vrijeme['vrijeme_key'].dt.year
dim_vrijeme['kvartal'] = dim_vrijeme['vrijeme_key'].dt.quarter
dim_vrijeme['dan_u_tjednu'] = dim_vrijeme['vrijeme_key'].dt.day_name()

dim_vrijeme.to_sql('dim_vrijeme', engine, if_exists='append', index=False)
print(f"dim_vrijeme: {len(dim_vrijeme)} redaka uneseno.")
print(f"Raspon: {dim_vrijeme['vrijeme_key'].min()} - {dim_vrijeme['vrijeme_key'].max()}")

## 5. TRANSFORM & LOAD - Punjenje tablice činjenica

In [ ]:
# 5a. TRANSFORM - Izračun metrike: vrijeme rješavanja u satima
df['vrijeme_rjesavanja_sati'] = (
    (df['issue_resolution_date'] - df['issue_created']).dt.total_seconds() / 3600
)
# Postavi negativne vrijednosti na 0 (greške u podacima)
df.loc[df['vrijeme_rjesavanja_sati'] < 0, 'vrijeme_rjesavanja_sati'] = 0
df['vrijeme_rjesavanja_sati'] = df['vrijeme_rjesavanja_sati'].round(2)

print(f"Metrika 'vrijeme_rjesavanja_sati' izračunata.")
print(f"  Prosjek: {df['vrijeme_rjesavanja_sati'].mean():.2f} h")
print(f"  Medijan: {df['vrijeme_rjesavanja_sati'].median():.2f} h")
print(f"  NULL (nema resolution_date): {df['vrijeme_rjesavanja_sati'].isna().sum()}")

In [ ]:
# 5b. TRANSFORM - Mapiranje na strane ključeve dimenzija
fact = df.copy()

# Mapiranje projekta
fact = fact.merge(
    db_projekti, 
    left_on='issue_proj', 
    right_on='naziv_projekta', 
    how='inner'
)
print(f"Nakon merge s dim_projekt: {len(fact)} redaka")

# Mapiranje reportera (inner - svaki ticket ima reportera)
fact = fact.merge(
    db_tehnicari, 
    left_on='issue_reporter', 
    right_on='ime_prezime', 
    how='inner'
).rename(columns={'tehnicar_key': 'reporter_key'})
print(f"Nakon merge s dim_tehnicar (reporter): {len(fact)} redaka")

# Mapiranje assigneea (LEFT - 46% ticketa nema assignee!)
fact = fact.merge(
    db_tehnicari, 
    left_on='issue_assignee', 
    right_on='ime_prezime', 
    how='left',
    suffixes=('', '_assignee')
).rename(columns={'tehnicar_key': 'assignee_key'})
print(f"Nakon merge s dim_tehnicar (assignee, LEFT): {len(fact)} redaka")
print(f"  Assignee NULL: {fact['assignee_key'].isna().sum()}")

# Mapiranje prioriteta i statusa
fact = fact.merge(
    db_ps, 
    left_on=['issue_priority', 'issue_status'], 
    right_on=['razina_prioriteta', 'naziv_statusa'],
    how='inner'
)
print(f"Nakon merge s dim_prioritet_status: {len(fact)} redaka")

In [ ]:
# 5c. TRANSFORM - Priprema vremenske dimenzije
fact['vrijeme_key'] = fact['issue_created'].dt.date

# 5d. LOAD - Odabir stupaca za fact tablicu prema DDL-u
fact_final = fact[[
    'id', 'projekt_key', 'reporter_key', 'assignee_key',
    'prioritet_status_key', 'vrijeme_key',
    'vrijeme_rjesavanja_sati', 'issue_comments_count'
]].copy()

# Rename i cast prema DDL shemi
fact_final = fact_final.rename(columns={
    'id': 'ticket_id',
    'issue_comments_count': 'broj_komentara'
})

# Cast ticket_id na int (originalno je float)
fact_final['ticket_id'] = fact_final['ticket_id'].astype(int)

# Assignee_key: NaN -> None za MySQL NULL
fact_final['assignee_key'] = fact_final['assignee_key'].astype('Int64')

print(f"Fact tablica pripremljena: {len(fact_final)} redaka")
print(f"Stupci: {list(fact_final.columns)}")
print(f"\nPrimjer prvih 5 redaka:")
print(fact_final.head())

In [ ]:
# 5e. LOAD - Unos u bazu
# Privremeno isključi FK provjere za brži bulk insert
with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.commit()

fact_final.to_sql(
    'fact_support_tickets', 
    engine, 
    if_exists='append', 
    index=False,
    chunksize=5000  # Batch insert za performanse
)

with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    conn.commit()

print(f"ETL proces uspješno završen!")
print(f"Uneseno {len(fact_final)} redaka u fact_support_tickets.")

## 6. Verifikacija

In [ ]:
# 6. Verifikacija unesenih podataka
print("=" * 50)
print("VERIFIKACIJA ETL PROCESA")
print("=" * 50)

with engine.connect() as conn:
    tables = {
        'dim_projekt': 'SELECT COUNT(*) FROM dim_projekt',
        'dim_tehnicar': 'SELECT COUNT(*) FROM dim_tehnicar',
        'dim_prioritet_status': 'SELECT COUNT(*) FROM dim_prioritet_status',
        'dim_vrijeme': 'SELECT COUNT(*) FROM dim_vrijeme',
        'fact_support_tickets': 'SELECT COUNT(*) FROM fact_support_tickets',
    }
    for name, query in tables.items():
        count = conn.execute(text(query)).scalar()
        print(f"  {name}: {count} redaka")

print(f"\nOčekivano u fact tablici: {len(fact_final)}")

# Provjera uzorka
sample = pd.read_sql("""
    SELECT 
        f.ticket_id,
        p.naziv_projekta,
        r.ime_prezime AS reporter,
        a.ime_prezime AS assignee,
        ps.razina_prioriteta,
        ps.naziv_statusa,
        f.vrijeme_key,
        f.vrijeme_rjesavanja_sati,
        f.broj_komentara
    FROM fact_support_tickets f
    JOIN dim_projekt p ON f.projekt_key = p.projekt_key
    JOIN dim_tehnicar r ON f.reporter_key = r.tehnicar_key
    LEFT JOIN dim_tehnicar a ON f.assignee_key = a.tehnicar_key
    JOIN dim_prioritet_status ps ON f.prioritet_status_key = ps.prioritet_status_key
    LIMIT 10
""", engine)

print(f"\nUzorak podataka iz data warehousea:")
print(sample.to_string(index=False))